In [1]:
pip install ffmpeg

  Preparing metadata (setup.py) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6083 sha256=f6b10ff359d4f62b8b22f30c2c62d7bee9c97ab5c1fc4ad0b2e79c82fc0d43a2
  Stored in directory: /root/.cache/pip/wheels/26/21/0c/c26e09dff860a9071683e279445262346e008a9a1d2142c4ad
Successfully built ffmpeg


In [2]:
from matplotlib.ticker import FuncFormatter
import matplotlib.ticker as mticker

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML

# --- Configuration ---
fps = 30
total_duration_sec = 10
pause_duration_sec = 2

# --- Labels ---
import numpy as np

labels = [
    "Mitarbeiterorientierte Führung",
    "Mitarbeiterorientierte Organisation",
    "Kund:innen",
    "Leistung",
    "Vision & Werte",
    "Nachhaltigkeit & Resilienz",
    "Work Life Flexibility",
    "Führung",
    "Team",
    "Agilität",
    "STARKE KULTUR"
]

# --- Data ---
gesamtbank = np.array([-0.8, -0.5, -0.4, -0.3, -1.1, -0.2, -0.1, 0.2, 2.0, -0.7, -0.6])
eigene_dynamik = np.array([-3.9, 3.2, -5.3, -1.6, 3.0, 0.2, -1.1, -0.6, 0.7, -0.9, -2.1])



# --- Reverse order (so STARKE KULTUR is on top) ---
labels = labels[::-1]
eigene_dynamik = eigene_dynamik[::-1]
gesamtbank = gesamtbank[::-1]

# --- Colors ---
dark_color = "#5c1f2b"
light_color = "#B2B2B2"

# --- Datasets ---
datasets_eigene = [np.array([v]) for v in eigene_dynamik]
datasets_gesamt = [np.array([v]) for v in gesamtbank]

# --- Animation setup ---
total_frames = int(total_duration_sec * fps)
pause_frames = int(pause_duration_sec * fps)
total_frames_with_pause = total_frames + pause_frames

# --- Figure setup ---
fig, ax = plt.subplots(figsize=(12, 5))
plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.25)
fig.patch.set_facecolor("#EAEAEF")
ax.set_facecolor("#EAEAEF")

# --- Spines ---
for side in ["right", "left", "top", "bottom"]:
    ax.spines[side].set_visible(False)

# --- Axis limits ---
ax.set_xlim(-20, 20.5)
ax.set_ylim(-0.5, len(labels) - 0.5)

# --- X grid (vertical lines) ---
x_ticks = np.arange(-20, 20.5, 10)
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_ticks, fontsize=8,color='gray')
ax.grid(True, axis='x', linestyle='-', color='gray', alpha=0.8, linewidth=0.2, zorder=1)

# --- Y grid (horizontal lines between bars) ---
y_positions = np.arange(len(labels))
# Grid lines *between* bar groups
y_grid_lines = np.arange(0.5, len(labels) - 0.5, 1)

ax.set_yticks(y_grid_lines)
ax.set_yticklabels([])
ax.grid(True, axis='y',
        linestyle=(0, (5, 5)),   # 5pt dash, 5pt gap
        color='gray',
        alpha=0.8,
        linewidth=0.2,
        zorder=1)

# --- Zero (center) line ---
ax.axvline(x=0, color="gray", linewidth=1.2, zorder=2)

# --- Bars ---
bars1 = ax.barh(y_positions + 0.15, [0]*len(labels),
                color=dark_color, height=0.3, zorder=3, label="Eigene Dynamik")

bars2 = ax.barh(y_positions - 0.15, [0]*len(labels),
                color=light_color, height=0.3, zorder=3, label="Gesamtbank")


# --- Static category labels (centered between horizontal lines) ---
for y, label in zip(y_positions, labels):
    ax.text(-19.5, y, label, va='center', ha='left', fontsize=9, color='gray', zorder=4)

# --- Text objects for animated values ---
value_texts_eigene = [ax.text(0, y + 0.13, "", va='center', ha='left', fontsize=8, zorder=4, color='#707070') for y in y_positions]
value_texts_gesamt = [ax.text(0, y - 0.18, "", va='center', ha='left', fontsize=6, zorder=4, color='#707070') for y in y_positions]

ax.tick_params(axis='x', which='major', length=0)  # hide main tick dashes
ax.tick_params(axis='y', which='major', length=0)  # hide main tick dashes

ax.legend(
    loc='upper center',          # position relative to axes
    bbox_to_anchor=(0.5, -0.25), # (x, y) = center below plot
    ncol=2,                      # 2 columns
    frameon=False,               # no border box
    fontsize=10,                 # adjust size as needed
    handlelength=1.8,            # length of color patch
    columnspacing=2.0            # spacing between legend entries
)

# --- Init ---
def init():
    for b1, b2 in zip(bars1, bars2):
        b1.set_width(0)
        b2.set_width(0)
    for t1, t2 in zip(value_texts_eigene, value_texts_gesamt):
        t1.set_text("")
        t2.set_text("")
    return (*bars1, *bars2, *value_texts_eigene, *value_texts_gesamt)

# --- Animate ---
def animate(i):
    if i >= total_frames:
        i = total_frames - 1
    progress = min(i / (total_frames - 1), 1.0)

    for (
        bar1, bar2, data1, data2, text1, text2
    ) in zip(bars1, bars2, datasets_eigene, datasets_gesamt, value_texts_eigene, value_texts_gesamt):
        value1 = data1[0] * progress
        value2 = data2[0] * progress

        bar1.set_x(min(0, value1))
        bar1.set_width(abs(value1))
        bar2.set_x(min(0, value2))
        bar2.set_width(abs(value2))

        # Texts
        text1.set_text(f"{value1:+.1f}" if abs(value1) > 0.05 else "")
        text2.set_text(f"{value2:+.1f}" if abs(value2) > 0.05 else "")

        gap = 0.2
        if value1 >= 0:
            text1.set_x(bar1.get_x() + bar1.get_width() + gap)
            text1.set_ha("left")
        else:
            text1.set_x(bar1.get_x() - gap)
            text1.set_ha("right")

        if value2 >= 0:
            text2.set_x(bar2.get_x() + bar2.get_width() + gap)
            text2.set_ha("left")
        else:
            text2.set_x(bar2.get_x() - gap)
            text2.set_ha("right")

    return (*bars1, *bars2, *value_texts_eigene, *value_texts_gesamt)

# --- Build animation ---
interval_ms = 1000 / fps
anim = FuncAnimation(
    fig, animate, init_func=init,
    frames=total_frames_with_pause, interval=interval_ms,
    blit=False, repeat=True
)

plt.draw()
plt.close(fig)
# --- Save MP4 ---
mp4_path = "1.mp4"
writer_mp4 = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt", "yuv420p"])
anim.save(mp4_path, writer=writer_mp4, dpi=300)
print(f"✅ Saved MP4: {mp4_path}")

plt.close(fig)

# --- Display as HTML5 video ---
HTML(anim.to_html5_video())


✅ Saved MP4: 1.mp4


In [4]:
from google.colab import files
files.download('1.mp4')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>